<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/NEW_TOPO_CBP_Muse_Glimmer_30B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://huggingface.co/datasets/SetFit/ag_news

https://huggingface.co/meta-models/Muse-Glimmer-30B

https://medium.com/ai-simplified-in-plain-english/solving-catastrophic-forgetting-the-topo-2026-certification-of-muse-glimmer-30b-bcd20545282c

In [1]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
# ==============================================================================
# BENCHMARK_TOPO_CBP_MUSE_GLIMMER_30B_AUDITED.py
#
# Controlled 4-Arm Continual Learning Benchmark on Muse-Glimmer-30B
#
# Full Audit & Protocol Alignment:
#   1. Explicit float32 cast on bfloat16 tensors prior to NumPy conversion
#      to eliminate PyTorch/NumPy scalar conversion errors.
#   2. Per-run atomic JSONL appending (benchmark_runs_stream.jsonl) ensuring
#      immediate persistence against hardware or runtime interrupts.
#   3. Sutton Continual Backpropagation operational tuning:
#      - Recycles 1 unit per 100 optimizer steps.
#      - Enforces a 200-step unit maturity window before re-eligibility.
#   4. Dynamic anchor shielding:
#      - Governed coordinate rows are explicitly masked out of the CBP recycling
#        pool so zero-gradient invariant anchors are never recycled.
#   5. Passive Step-0 baseline reference capture for all arms to record true drift:
#      - Arm 1 (Standard AdamW): Unconstrained gradient displacement.
#      - Arm 2 (Pure CBP): Displacement under Sutton utility dynamics.
#      - Arm 3 (Topo-CBP Prime): Invariance verification via coordinate restoration.
#      - Arm 4 (Composite Control): Unpinned prime-row displacement.
#   6. Unclipped, signed retention calculations alongside lower-triangular
#      retention matrices exported directly for independent empirical review.
# ==============================================================================

import gc
import hashlib
import json
import math
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForMultimodalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    logging as hf_logging,
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
hf_logging.set_verbosity_error()

# ==============================================================================
# 1. CONFIGURATION & EXPERIMENTAL CONSTANTS
# ==============================================================================
MODEL_ID = 'meta-models/Muse-Glimmer-30B'
HIDDEN_SIZE = 6656
SHARED_DIM = 512
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SAFETY_CONSTANT = 0.9785142874
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
COMPOSITE_ANCHORS = [4, 6, 8, 9, 10, 12]
SEEDS = [42, 123, 999]

MAX_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 5
BATCH_SIZE = 8
LEARNING_RATE = 1e-4

# Sutton CBP Operational Constants
CBP_REPLACEMENT_INTERVAL = 100  # Replacement interval in optimizer steps
CBP_REPLACEMENT_UNITS = 1       # Units recycled per interval
CBP_MATURITY_STEPS = 200        # Minimum age required before eligibility
CBP_DECAY = 0.99

JSONL_STREAM_PATH = "benchmark_runs_stream.jsonl"

print(f"[Init] Initializing Benchmark on {DEVICE} | Shared Bottleneck Dimension: {SHARED_DIM}")

# ==============================================================================
# 2. DATASET STREAMING (SetFit/ag_news)
# ==============================================================================
print("Loading 'SetFit/ag_news' continual dataset splits...")
raw_train = load_dataset('SetFit/ag_news', split='train[:3000]')
raw_test = load_dataset('SetFit/ag_news', split='test[:600]')

class TaskDataset(Dataset):
    def __init__(self, dataset, tokenizer, allowed_labels, label_mapping, max_length=64):
        self.filtered_data = [
            (text, label_mapping[label])
            for text, label in zip(dataset['text'], dataset['label'])
            if label in allowed_labels
        ]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.filtered_data)

    def __getitem__(self, idx):
        text, label = self.filtered_data[idx]
        enc = self.tokenizer(
            text, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

task_configs = {
    'A': {'allowed': [0, 1], 'mapping': {0: 0, 1: 1}},
    'B': {'allowed': [2, 3], 'mapping': {2: 0, 3: 1}},
    'C': {'allowed': [0, 3], 'mapping': {0: 0, 3: 1}}
}

def build_loaders(tokenizer):
    train_loaders, test_loaders = {}, {}
    for k, cfg in task_configs.items():
        train_loaders[k] = DataLoader(
            TaskDataset(raw_train, tokenizer, cfg['allowed'], cfg['mapping']),
            batch_size=BATCH_SIZE,
            shuffle=True
        )
        test_loaders[k] = DataLoader(
            TaskDataset(raw_test, tokenizer, cfg['allowed'], cfg['mapping']),
            batch_size=BATCH_SIZE,
            shuffle=False
        )
    return train_loaders, test_loaders

# ==============================================================================
# 3. CONTINUAL MODEL WITH SHARED ADAPTER (FP32 INTERMEDIATE MANIFOLD)
# ==============================================================================
class MuseGlimmer_TaskAwareModel(nn.Module):
    def __init__(self, base_model: nn.Module, hidden_size: int = HIDDEN_SIZE, shared_dim: int = SHARED_DIM):
        super().__init__()
        self.base_model = base_model
        self.hidden_size = hidden_size

        # Intermediate bottleneck adapter in float32 for clean gradient registration
        self.shared_adapter = nn.Linear(hidden_size, shared_dim, bias=False, dtype=torch.float32).to(DEVICE)
        self.act = nn.GELU()

        # Task classification heads
        self.classifier_A = nn.Linear(shared_dim, 2, dtype=torch.float32).to(DEVICE)
        self.classifier_B = nn.Linear(shared_dim, 2, dtype=torch.float32).to(DEVICE)
        self.classifier_C = nn.Linear(shared_dim, 2, dtype=torch.float32).to(DEVICE)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        with torch.no_grad():
            outputs = self.base_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                return_dict=True
            )
            hidden_states = outputs.hidden_states[-1]
            if attention_mask is not None:
                seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
                batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
                last_hidden = hidden_states[batch_idx, seq_lens, :].float()
            else:
                last_hidden = hidden_states[:, -1, :].float()

        shared_rep = self.act(self.shared_adapter(last_hidden))
        head = getattr(self, f"classifier_{self.current_task}")
        return head(shared_rep)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task

# ==============================================================================
# 4. ACTIVE TOPOLOGICAL GOVERNOR
# ==============================================================================
class TopoCBPGovernor:
    def __init__(self, target_tensor: nn.Parameter, anchors: list, lambda_const: float = SAFETY_CONSTANT):
        self.target_tensor = target_tensor
        self.anchors = [a for a in anchors if a < target_tensor.shape[0]]
        self.lambda_const = lambda_const
        self.snapshot = {idx: target_tensor[idx].clone().detach() for idx in self.anchors}
        self.state_hash = self._compute_hash()

    def _compute_hash(self):
        stacked = torch.cat([self.snapshot[idx].flatten() for idx in self.anchors])
        return hashlib.sha256(stacked.float().cpu().numpy().tobytes()).hexdigest()

    def phase1_pre_step_masking(self):
        if self.target_tensor.grad is not None:
            self.target_tensor.grad.mul_(self.lambda_const)
            for idx in self.anchors:
                self.target_tensor.grad[idx].zero_()

    def phase2_post_step_restoration(self):
        with torch.no_grad():
            for idx in self.anchors:
                self.target_tensor[idx].copy_(self.snapshot[idx])

# ==============================================================================
# 5. SUTTON CONTINUAL BACKPROP ENGINE (WITH ANCHOR EXCLUSION & MATURITY)
# ==============================================================================
class SuttonCBPEngine:
    def __init__(
        self,
        target_layer: nn.Linear,
        protected_anchors: list = None,
        replacement_interval: int = CBP_REPLACEMENT_INTERVAL,
        units_to_replace: int = CBP_REPLACEMENT_UNITS,
        maturity_steps: int = CBP_MATURITY_STEPS,
        decay: float = CBP_DECAY,
    ):
        self.target_layer = target_layer
        self.protected_anchors = set(protected_anchors or [])
        self.replacement_interval = replacement_interval
        self.units_to_replace = units_to_replace
        self.maturity_steps = maturity_steps
        self.decay = decay

        self.num_units = target_layer.weight.shape[0]
        self.utility = torch.zeros(self.num_units, device=DEVICE, dtype=torch.float32)
        self.age = torch.zeros(self.num_units, dtype=torch.long, device=DEVICE)
        self.total_steps = 0

    def step_utility_tracking(self):
        self.total_steps += 1
        self.age += 1
        param = self.target_layer.weight

        if param.grad is not None:
            grad_contrib = torch.abs(param.grad * param.data).sum(dim=1).detach()
            self.utility.mul_(self.decay).add_(grad_contrib, alpha=(1.0 - self.decay))

        if self.total_steps % self.replacement_interval == 0:
            self._recycle_units()

    def _recycle_units(self):
        param = self.target_layer.weight

        # Filter units by maturity
        eligible_mask = self.age >= self.maturity_steps

        # Shield anchored coordinates from recycling
        for anchor_idx in self.protected_anchors:
            if anchor_idx < self.num_units:
                eligible_mask[anchor_idx] = False

        eligible_indices = torch.where(eligible_mask)[0]
        if len(eligible_indices) == 0:
            return

        k = min(self.units_to_replace, len(eligible_indices))
        lowest_relative = torch.topk(self.utility[eligible_indices], k=k, largest=False).indices
        recycled_indices = eligible_indices[lowest_relative]

        with torch.no_grad():
            fan_in = param.shape[1]
            param.data[recycled_indices] = torch.randn(
                len(recycled_indices), fan_in, device=DEVICE, dtype=param.dtype
            ) * (1.0 / math.sqrt(fan_in))

            self.utility[recycled_indices] = self.utility.mean()
            self.age[recycled_indices] = 0

# ==============================================================================
# 6. EVALUATION HELPER
# ==============================================================================
def evaluate_task(model, task_name, dataloader):
    model.switch_task(task_name)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(logits, dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total if total > 0 else 0.0

# ==============================================================================
# 7. EXPERIMENTAL RUNNER
# ==============================================================================
def run_arm(base_model, tokenizer, arm_cfg, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

    train_loaders, test_loaders = build_loaders(tokenizer)
    model = MuseGlimmer_TaskAwareModel(base_model, HIDDEN_SIZE, SHARED_DIM)

    # Step-0 passive reference snapshot of prime rows across all arms
    prime_ref_snapshot = {
        idx: model.shared_adapter.weight[idx].clone().detach() for idx in PRIME_ANCHORS
    }

    governor = TopoCBPGovernor(
        model.shared_adapter.weight, anchors=arm_cfg['anchors'], lambda_const=SAFETY_CONSTANT
    ) if arm_cfg['gov'] else None

    cbp_engine = SuttonCBPEngine(
        model.shared_adapter, protected_anchors=arm_cfg['anchors']
    ) if arm_cfg['cbp'] else None

    trainable_params = [
        model.shared_adapter.weight,
        model.classifier_A.weight, model.classifier_A.bias,
        model.classifier_B.weight, model.classifier_B.bias,
        model.classifier_C.weight, model.classifier_C.bias
    ]
    optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE)

    task_names = ['A', 'B', 'C']
    retention_matrix = np.zeros((3, 3))

    for i, task_name in enumerate(task_names):
        model.switch_task(task_name)
        best_acc = 0.0
        patience_cnt = 0

        for epoch in range(MAX_EPOCHS):
            model.train()
            for batch in train_loaders[task_name]:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)

                logits = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = F.cross_entropy(logits, labels)
                loss.backward()

                if governor is not None:
                    governor.phase1_pre_step_masking()

                optimizer.step()

                if governor is not None:
                    governor.phase2_post_step_restoration()

                if cbp_engine is not None:
                    cbp_engine.step_utility_tracking()

            val_acc = evaluate_task(model, task_name, test_loaders[task_name])
            if val_acc > best_acc:
                best_acc = val_acc
                patience_cnt = 0
            else:
                patience_cnt += 1
                if patience_cnt >= EARLY_STOPPING_PATIENCE:
                    break

        for j, eval_task in enumerate(task_names):
            if j <= i:
                retention_matrix[i, j] = evaluate_task(model, eval_task, test_loaders[eval_task])

    # Unclipped, signed backward transfer (forgetting)
    fgt_A_signed = (retention_matrix[0, 0] - retention_matrix[2, 0]) * 100
    fgt_B_signed = (retention_matrix[1, 1] - retention_matrix[2, 1]) * 100
    mean_fgt_signed = (fgt_A_signed + fgt_B_signed) / 2.0

    prime_row_drift = 0.0
    with torch.no_grad():
        for idx in PRIME_ANCHORS:
            delta = torch.max(torch.abs(model.shared_adapter.weight[idx] - prime_ref_snapshot[idx])).cpu().item()
            if delta > prime_row_drift:
                prime_row_drift = delta

    run_record = {
        "Arm": arm_cfg['name'],
        "Seed": seed,
        "Anchors": str(arm_cfg['anchors']),
        "Retention_Matrix": retention_matrix.tolist(),
        "Task_A_Initial": retention_matrix[0, 0] * 100,
        "Task_A_Final": retention_matrix[2, 0] * 100,
        "Task_A_FGT": fgt_A_signed,
        "Task_B_Initial": retention_matrix[1, 1] * 100,
        "Task_B_Final": retention_matrix[2, 1] * 100,
        "Task_B_FGT": fgt_B_signed,
        "Mean_FGT": mean_fgt_signed,
        "Task_C_Plasticity": retention_matrix[2, 2] * 100,
        "Prime_Row_Drift": prime_row_drift
    }

    # Atomic per-run serialization to JSONL
    with open(JSONL_STREAM_PATH, "a") as f:
        f.write(json.dumps(run_record) + "\n")

    del model, optimizer
    torch.cuda.empty_cache()
    gc.collect()

    return run_record

# ==============================================================================
# 8. EXECUTION CONTROLLER
# ==============================================================================
if __name__ == '__main__':
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    print(f"Loading {MODEL_ID} (4-bit NF4)...")
    base_model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        max_memory={0: "22GB", "cpu": "30GB"},
        dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    base_model.config.use_cache = False
    for p in base_model.parameters():
        p.requires_grad = False

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    arms = [
        {"name": "Arm 1: Standard AdamW (Baseline)", "gov": False, "cbp": False, "anchors": []},
        {"name": "Arm 2: Pure CBP (Sutton)",         "gov": False, "cbp": True,  "anchors": []},
        {"name": "Arm 3: Topo-CBP (Prime Basis)",     "gov": True,  "cbp": True,  "anchors": PRIME_ANCHORS},
        {"name": "Arm 4: Composite Control",         "gov": True,  "cbp": True,  "anchors": COMPOSITE_ANCHORS}
    ]

    print("\n" + "=" * 80)
    print("STARTING CONTROLLED 4-ARM CONTINUAL LEARNING BENCHMARK")
    print("=" * 80)

    results = []

    # 1. Arm 1 Seed 123 Sanity Gate
    print("\n>>> EXECUTING ARM 1 GATE (Verifying Baseline Degradation on Shared Weights)...")
    gate_res = run_arm(base_model, tokenizer, arms[0], seed=123)
    print(f"Gate Complete -> Task A FGT: {gate_res['Task_A_FGT']:.2f}% | Mean FGT: {gate_res['Mean_FGT']:.2f}% | Prime Drift: {gate_res['Prime_Row_Drift']:.6f}")

    if gate_res['Mean_FGT'] <= 5.0:
        print(f"\n⚠️ WARNING: Low baseline forgetting observed ({gate_res['Mean_FGT']:.2f}%).")
        print("Verify task interference and gradient flow through the shared adapter before proceeding.")
    else:
        print("✅ Baseline forgetting verified. Commencing full sweep across remaining arms and seeds...")

    results.append(gate_res)

    # 2. Complete Experimental Sweep
    for arm in arms:
        for s in SEEDS:
            if arm['name'] == arms[0]['name'] and s == 123:
                continue

            print(f"\nExecuting {arm['name']} | Seed {s}...")
            res = run_arm(base_model, tokenizer, arm, seed=s)
            results.append(res)
            print(f"Completed -> Task A Retention: {res['Task_A_Final']:.2f}% | Mean FGT: {res['Mean_FGT']:.2f}% | Prime Drift: {res['Prime_Row_Drift']:.6f}")

    # ==============================================================================
    # 9. CONSOLIDATE TELEMETRY & ARTIFACT EXPORT
    # ==============================================================================
    df = pd.DataFrame(results)
    summary = df.groupby('Arm').agg(
        Mean_FGT_Avg=('Mean_FGT', 'mean'),
        Mean_FGT_Std=('Mean_FGT', 'std'),
        Task_A_Retention=('Task_A_Final', 'mean'),
        Task_C_Plasticity=('Task_C_Plasticity', 'mean'),
        Avg_Prime_Drift=('Prime_Row_Drift', 'mean')
    ).reset_index()

    print("\n" + "=" * 80)
    print("FINAL RETENTION & PLASTICITY MATRIX AUDIT")
    print("=" * 80)
    print(summary.to_string(index=False))

    df.to_csv("benchmark_muse_glimmer_30b_telemetry.csv", index=False)
    summary.to_csv("benchmark_muse_glimmer_30b_summary.csv", index=False)

    with open("retention_matrices_raw.json", "w") as f:
        json.dump([
            {
                "Arm": r["Arm"],
                "Seed": r["Seed"],
                "Anchors": r["Anchors"],
                "Retention_Matrix": r["Retention_Matrix"]
            } for r in results
        ], f, indent=2)

    print("\n✅ Sweep execution complete. Generated audit files:")
    print("  -> benchmark_runs_stream.jsonl (Raw atomic streaming log)")
    print("  -> benchmark_muse_glimmer_30b_telemetry.csv")
    print("  -> benchmark_muse_glimmer_30b_summary.csv")
    print("  -> retention_matrices_raw.json")

[Init] Initializing Benchmark on cuda | Shared Bottleneck Dimension: 512
Loading 'SetFit/ag_news' continual dataset splits...
Loading meta-models/Muse-Glimmer-30B (4-bit NF4)...


Loading weights:   0%|          | 0/1436 [00:00<?, ?it/s]


STARTING CONTROLLED 4-ARM CONTINUAL LEARNING BENCHMARK

>>> EXECUTING ARM 1 GATE (Verifying Baseline Degradation on Shared Weights)...
Gate Complete -> Task A FGT: 14.10% | Mean FGT: 11.22% | Prime Drift: 0.036578
✅ Baseline forgetting verified. Commencing full sweep across remaining arms and seeds...

Executing Arm 1: Standard AdamW (Baseline) | Seed 42...
Completed -> Task A Retention: 90.06% | Mean FGT: 6.01% | Prime Drift: 0.029588

Executing Arm 1: Standard AdamW (Baseline) | Seed 999...
Completed -> Task A Retention: 62.50% | Mean FGT: 17.40% | Prime Drift: 0.028352

Executing Arm 2: Pure CBP (Sutton) | Seed 42...
Completed -> Task A Retention: 88.78% | Mean FGT: 3.83% | Prime Drift: 0.022295

Executing Arm 2: Pure CBP (Sutton) | Seed 123...
Completed -> Task A Retention: 85.90% | Mean FGT: 6.40% | Prime Drift: 0.031969

Executing Arm 2: Pure CBP (Sutton) | Seed 999...
